In [9]:
import sys, os
sys.path.append(os.path.abspath('/afs/cern.ch/user/a/anunezde/bamboodev/hh/Bamboo_setup/src'))
from post_processing.NN.DNNManager import DNNManager
from post_processing import References as Refs
from pathlib import Path
import pandas as pd
import uproot
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)  # Set a larger width to fit the editor window
# pd.set_option('display.max_colwidth', None)  # Allow columns to be fully displayed

In [19]:
Z_OUTPUT_eos = Path('/eos/user/a/anunezde/Z_OUTPUT_eos')
BAMBOO_SETUP = DNNManager.BAMBOO_SETUP

workdir = Z_OUTPUT_eos / '2022_even_0822' / 'LLR_and_vars_4o5'
sel_name = 'SL_res_2b_x'
models_yml = 'NN_DNNManager_test.yml'
total_inputs = 'Total_inputs/vars40.txt'
DNNManagerdir =  'TEST_TEST'

In [15]:
class DNNManager_focus(DNNManager):

    def load_data(self) -> list[pd.DataFrame]:
        print(f"\nLoading data ...")

        processes_available = Refs._find_processes(self.RESULTSDIR)
        root_files_available = Refs._find_root_files(self.RESULTSDIR)

        if self.total_inputs is not None:
            with open(self.POSTPROCESSING_NN_FOLDER / self.total_inputs) as file:
                branches = [line.strip() for line in file]
            array_extractor = lambda upfile, sel_name: upfile[sel_name].arrays(branches, library="pd")
        else:
            array_extractor = lambda upfile, sel_name: upfile[sel_name].arrays(library="pd")

        df_list = []
        for process in processes_available:
            process_df = pd.DataFrame()
            process_files = [file for file in root_files_available if file.stem in Refs.PROCESSES_FILES[process]]
            for file in process_files:
                upfile = uproot.open(file)
                upfile_df = array_extractor(upfile, self.sel_name)[:1000000]
                upfile_df['File'] = file.stem
                print(f'Number of events read from {file.stem}: {len(upfile_df)}')
                process_df = pd.concat([process_df, upfile_df], ignore_index=True)
            process_df['Process'] = process
            df_list.append(process_df)

        total_df = pd.concat(df_list, ignore_index=True)
        total_df = pd.get_dummies(total_df, columns=['Process'])

        total_df.reset_index(inplace=True)
        total_df.sort_values(by=['event', 'index'], inplace=True)
        total_df.drop(columns='index', inplace=True)

        print(f"Total_df:\n{total_df}")
        
        return total_df

In [20]:
manager = DNNManager_focus(workdir=workdir, sel_name=sel_name, models_yml=models_yml, total_inputs=total_inputs, DNNManagerdir=DNNManagerdir)
total_df = manager.load_data()

DNN Manager instantiated:
	WORKDIR:/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_0822/LLR_and_vars_4o5
	RESULTSDIR:/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_0822/LLR_and_vars_4o5/results
	DNNMANAGERDIR:/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_0822/LLR_and_vars_4o5/TEST_TEST

Loading data ...


KeyInFileError: not found: 'run'
in file /eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_0822/LLR_and_vars_4o5/results/DY_dl_mll_10to50.root
in object /SL_res_2b_x;1

In [18]:
files = total_df['File'].unique().tolist()
print(f"All files: {files}")
files_with_no_duplicates = []
for file_name in files:
    file_events = total_df[total_df['File'] == file_name]
    duplicate_file_event_nums = file_events[file_events.duplicated(subset='event', keep=False)]
    if duplicate_file_event_nums.empty:
        files_with_no_duplicates.append(file_name)
    else:
        print(f"\nDuplicate event numbers found for {file_name}:")
        print(duplicate_file_event_nums[['event']].drop_duplicates())

print(f"\nNo duplicates found in: {files_with_no_duplicates}")

All files: ['tbarWplus_dl', 'TTbar_sl', 'bbWW_sl', 'TTbar_dl', 'bbWW_dl', 'tbarWplus_sl', 'bbtautau', 'tWminus_dl', 'tWminus_sl', 'WW', 'DY_dl_mll_50_2J', 'WZ', 'Wjets_2J', 'DY_dl_mll_10to50', 'ZZ', 'DY_dl_mll_50_1J', 'Wjets_1J', 'DY_dl_mll_50_0J', 'Wjets_0J']

Duplicate event numbers found for tbarWplus_dl:
          event
165218    19566
167403    22412
163872    29832
163873    29986
163874    30194
...         ...
164282  2422496
170552  2423626
170587  2431606
170630  2444126
172993  2473014

[846 rows x 1 columns]

Duplicate event numbers found for TTbar_sl:
            event
1313109        22
1313110        62
1313111        68
1313112        96
1313113       114
...           ...
1211987  72971462
1211988  72971488
1211989  72971492
1211990  72971502
1211991  72971524

[265878 rows x 1 columns]

Duplicate event numbers found for TTbar_dl:
           event
319613      1846
319617      2048
483172      2748
443700      5464
443813     13638
...          ...
345516  24981362
42093